# Evaluate Notebook

- Source: `src/evaluate.py`
- 목적: 원본 파이썬 파일을 단계별로 실행/설명하기 위한 노트북 버전
- 실행 방법: 위에서 아래로 순서대로 실행


In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    # 노트북이 다른 경로에서 열렸을 때 프로젝트 루트 자동 탐색
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / 'src').exists() and (parent / 'configs').exists():
            PROJECT_ROOT = parent
            break
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')


## Step 1. Setup and Imports

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
"""테스트셋 평가 + Confusion Matrix 저장.

이 스크립트가 하는 일
─────────────────
1. checkpoints/best_<task>.pth 로드 (학습때 저장된 가장 좋은 모델)
2. test split CSV를 읽어 데이터로더 생성
3. Accuracy / macro-F1 / 클래스별 F1 출력
4. Confusion Matrix 이미지를 docs/<task>_confusion_matrix.png 저장

사용 예시
─────────
    # 로스팅 모델 테스트
    python -m src.evaluate --config configs/default.yaml

    # 결점두 모델 테스트
    python -m src.evaluate --config configs/defect.yaml

    # 특정 checkpoint 지정 (기본값은 best_<task>.pth)
    python -m src.evaluate --config configs/defect.yaml --ckpt checkpoints/my_model.pth
"""
from __future__ import annotations
import argparse
from pathlib import Path

import torch
import numpy as np
import matplotlib
matplotlib.use("Agg")  # GUI 없이 파일만 저장
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from torch.utils.data import DataLoader

from src.utils.config import load_config
from src.utils.seed import set_seed
from src.dataset import SingleTaskDataset, get_transforms
from src.model import CoffeeClassifier


## Step 2. Function: load_checkpoint

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def load_checkpoint(path: str | Path, device: str):
    # checkpoint 형식을 하나로 맞춰서 사용하는 helper.
    # dict 전체가 들어오든 state_dict만 들어오든 아래 코드가 공통으로 처리할 수 있다.
    ckpt = torch.load(path, map_location=device, weights_only=False)
    state = ckpt["model_state_dict"] if isinstance(ckpt, dict) and \
            "model_state_dict" in ckpt else ckpt
    return ckpt, state


## Step 3. Function: main

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", required=True)
    ap.add_argument("--ckpt", default=None)
    ap.add_argument("--out_dir", default="docs")
    args = ap.parse_args()

    cfg = load_config(args.config)
    set_seed(cfg["seed"])
    device = "cuda" if torch.cuda.is_available() else "cpu"

    task = cfg["task"]
    classes = cfg["classes"]
    # 학습 때와 같은 클래스 순서를 다시 만들어준다.
    c2i = {c: i for i, c in enumerate(classes)}
    idx_to_class = {i: c for c, i in c2i.items()}

    ckpt_path = args.ckpt or f"{cfg['paths']['ckpt_dir']}/best_{task}.pth"
    ckpt, state = load_checkpoint(ckpt_path, device)

    # 체크포인트에 저장된 config가 있으면 backbone은 그걸 우선 사용
    # checkpoint 안에 저장된 backbone 정보를 우선 사용한다.
    # 그래야 config를 나중에 바꿨더라도 예전 weight를 안전하게 읽을 수 있다.
    ckpt_cfg = ckpt.get("config") if isinstance(ckpt, dict) else None
    backbone = (ckpt_cfg or cfg)["model"]["name"]
    print(f"[i] using backbone from checkpoint: {backbone}")

    model = CoffeeClassifier(
        backbone=backbone, n_classes=len(classes),
        pretrained=False, dropout=cfg["model"]["dropout"],
        hidden=cfg["model"]["hidden"],
    ).to(device)
    model.load_state_dict(state)
    model.eval()

    # test split은 절대 shuffle하지 않는다.
    # 순서가 바뀌어도 metric은 같지만, 디버깅할 때 원본 CSV 순서와 맞추기 편하다.
    test_ds = SingleTaskDataset(
        csv_path=cfg["data"]["test_csv"], task=task, class_to_idx=c2i,
        transform=get_transforms(task=task, train=False,
                                 img_size=cfg["data"]["img_size"]),
    )
    loader = DataLoader(test_ds, batch_size=cfg["train"]["batch_size"],
                        shuffle=False, num_workers=cfg["train"]["num_workers"])

    ys, ps = [], []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            ps.append(model(x).argmax(1).cpu())
            ys.append(y)
    y_true = torch.cat(ys).numpy()
    y_pred = torch.cat(ps).numpy()

    # 전체 정확도와 클래스별 균형 성능을 함께 보기 위해 macro F1도 같이 출력한다.
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro")
    print(f"\n[TEST] acc={acc:.4f}  macro_f1={f1:.4f}\n")
    print(classification_report(y_true, y_pred,
                                target_names=[idx_to_class[i] for i in range(len(classes))],
                                digits=4))

    cm = confusion_matrix(y_true, y_pred)
    out = Path(args.out_dir); out.mkdir(parents=True, exist_ok=True)
    n = len(classes)
    # 클래스 수가 많아질수록 confusion matrix 그림 크기를 조금 키운다.
    side = max(6, int(0.55 * n) + 4)
    plt.figure(figsize=(side, side - 1))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=classes, yticklabels=classes,
                annot_kws={"size": 8 if n > 8 else 10})
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.xlabel("predicted"); plt.ylabel("true")
    plt.title(f"Confusion Matrix ({task})  acc={acc:.3f}  f1={f1:.3f}")
    plt.tight_layout()
    cm_path = out / f"confusion_matrix_{task}.png"
    plt.savefig(cm_path, dpi=120)
    print(f"[OK] saved {cm_path}")


## Step 4. Run Entry Point

이 셀은 원본 코드의 해당 블록을 그대로 옮긴 단계입니다.


In [ ]:
if __name__ == "__main__":
    main()


## 실행 파라미터 가이드

- 이 파일은 원래 CLI 인자(argparse) 기반으로 동작합니다.
- 노트북에서는 인자 대신 아래처럼 변수 셀을 만들어 실행하세요.


In [ ]:
# 예시 파라미터 셀
CONFIG_PATH = 'configs/default.yaml'
CKPT_PATH = None
